In [35]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
# Load environment variables from .env
load_dotenv()
llm = ChatGroq(
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile"
)
response = llm.invoke("The first person to land on the Moon was ...")
print(response.content)

The first person to land on the Moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the Moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the Moon.


In [36]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/senior-infrastructure-engineer/job/R-87210")
page_data = loader.load().pop().page_content
print(page_data)





















Senior Infrastructure Engineer














































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous 

In [37]:
from langchain_core.prompts import PromptTemplate
prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):
        """
)
chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [38]:
from langchain_core.output_parsers import JsonOutputParser
json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Senior Infrastructure Engineer',
 'experience': '4+ years of experience in a network security engineering role',
 'skills': ['enterprise level Security acumen',
  'experience working on Palo Alto Next Generation Firewalls',
  'managerial experience',
  'Understanding of network design and operations across full stack technologies',
  'Knowledge of network threat prevention methodologies and protocols',
  'Experience with Network Access Control systems such as Cisco ISE',
  'Hands-on experience with traffic decryption and privacy regulations',
  'Proficiency with network automation tools and scripting languages',
  'Strong communication and interpersonal skills',
  'Excellent problem-solving skills and ability to manage multiple priorities'],
 'description': 'Provide expert-level technical engineering and design for enterprise network security infrastructure, working complex firewall deployments, handling high-level tier-3 troubleshooting, and driving innovation in advanced cl

In [39]:
type(json_res)

dict

In [40]:
import pandas as pd
df = pd.read_csv("app/resource/my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [41]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [45]:
links = collection.query(query_texts=job['skills'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/ios-ar-portfolio'},
  {'links': 'https://example.com/java-portfolio'}],
 [{'links': 'https://example.com/flutter-portfolio'},
  {'links': 'https://example.com/kotlin-android-portfolio'}],
 [{'links': 'https://example.com/magento-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/full-stack-js-portfolio'},
  {'links': 'https://example.com/xamarin-portfolio'}],
 [{'links': 'https://example.com/kotlin-android-portfolio'},
  {'links': 'https://example.com/flutter-portfolio'}],
 [{'links': 'https://example.com/ios-ar-portfolio'},
  {'links': 'https://example.com/xamarin-portfolio'}],
 [{'links': 'https://example.com/ios-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/devops-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/ios-portfolio'}],
 [{'lin

In [44]:
job = json_res
job['skills']

['enterprise level Security acumen',
 'experience working on Palo Alto Next Generation Firewalls',
 'managerial experience',
 'Understanding of network design and operations across full stack technologies',
 'Knowledge of network threat prevention methodologies and protocols',
 'Experience with Network Access Control systems such as Cisco ISE',
 'Hands-on experience with traffic decryption and privacy regulations',
 'Proficiency with network automation tools and scripting languages',
 'Strong communication and interpersonal skills',
 'Excellent problem-solving skills and ability to manage multiple priorities']

In [46]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}

        ### INSTRUCTION:
        You are Rohan, a business development executive at XYZ. XYZ is an AI & Software Consulting company dedicated to facilitating the seamless integration of business processes through automated tools.
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability,
        process optimization, cost reduction, and heightened overall efficiency.
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of XYZ
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase XYZ's portfolio: {link_list}
        Remember you are Rohan, BDE at XYZ.
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):

        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Expert Solutions for Enterprise Network Security Infrastructure

Dear Hiring Manager,

I came across the job description for a Senior Infrastructure Engineer at your organization, and I am excited to introduce XYZ, an AI & Software Consulting company that can fulfill your needs. With our expertise in providing tailored solutions, we can empower your enterprise to achieve scalability, process optimization, cost reduction, and heightened overall efficiency.

Our team at XYZ has extensive experience in designing and implementing enterprise-level network security infrastructure, including complex firewall deployments and network threat prevention methodologies. We have hands-on experience with traffic decryption, privacy regulations, and network automation tools, which aligns with your requirements. Our proficiency in scripting languages and strong communication skills ensure seamless integration and effective collaboration.

To demonstrate our capabilities, I would like to highli